In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_phone = pd.read_csv("../data/processed/phone_feature_extracted_train(4s).csv")
df_watch = pd.read_csv("../data/processed/watch_feature_extracted_train(4s).csv")

In [ ]:
df_phone

In [ ]:
df_phone.skew(numeric_only=True)

In [ ]:
X_phone = df_phone.drop(columns="activity")
y_phone = df_phone["activity"]

X_watch = df_watch.drop(columns="activity")
y_watch = df_watch["activity"]

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score


def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)

    max_depth = trial.suggest_int("max_depth", 5, 50)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 5)

    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    criterion = trial.suggest_categorical("criterion", ["entropy", "gini"])

    max_features = trial.suggest_categorical(
        "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]
    )

    max_samples = trial.suggest_categorical("max_samples", [0.9, 1])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        max_features=max_features,
        max_samples=max_samples,
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
    )

    score = cross_val_score(
        model,
        X_phone,
        y_phone,
        cv=3,
        scoring=make_scorer(f1_score, average="macro"),
        n_jobs=-1,
    ).mean()

    return score


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
)


study.optimize(objective, n_trials=50)

In [ ]:
model = RandomForestClassifier(
    n_estimators=322,
    max_depth=40,
    min_samples_leaf=1,
    min_samples_split=2,
    criterion="entropy",
    n_jobs=-1,
)

score = cross_val_score(
    model,
    X_phone,
    y_phone,
    cv=3,
    scoring=make_scorer(f1_score, average="macro"),
    n_jobs=-1,
).mean()

score

In [ ]:
model.fit(X_phone, y_phone)

In [ ]:
import joblib

joblib.dump(model, '../models/rfc_phone.pkl')

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

dc = DummyClassifier(strategy="most_frequent", random_state=42)

dc.fit(X_phone, y_phone)

y_pred_dummy = dc.predict(X_test_phone)

# 4. Calculate the F1-score (weighted for multi-class)
f1_dummy = f1_score(y_test_phone, y_pred_dummy, average="weighted")

print(f"Dummy Classifier F1-score (weighted): {f1_dummy:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_predict

y_preds_phone = cross_val_predict(
    model,
    X_train_phone,
    y_train_phone,
    cv=3,
    n_jobs=-1,
)

In [ ]:
y_train_phone = le.inverse_transform(y_train_phone)
y_preds_phone = le.inverse_transform(y_preds_phone)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


print(classification_report(y_train_phone, y_preds_phone))


cm = confusion_matrix(y_train_phone, y_preds_phone)

fig, ax = plt.subplots(figsize=(14, 12))

# Plot the matrix using the display class
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap="Blues", values_format="d")

disp.ax_.set_xticklabels(
    disp.ax_.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=14,
)

disp.ax_.set_yticklabels(disp.ax_.get_yticklabels(), fontsize=14)

disp.ax_.set_title(
    "Confusion Matrix - Phone Sensor (Cross-Validation)", fontsize=18, fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
y_train_probs = cross_val_predict(
    model,
    X_train_phone,
    y_train_phone,
    cv=3,
    method="predict_proba",
    n_jobs=-1,
)

In [ ]:
ACTIVITIES = {
    "A": "Walking",
    "B": "Jogging",
    "C": "Stairs",
    "D": "Sitting",
    "E": "Standing",
    "F": "Typing",
    "G": "Brushing Teeth",
    "H": "Eating Soup",
    "I": "Eating Chips",
    "J": "Eating Pasta",
    "K": "Drinking from Cup",
    "L": "Eating Sandwich",
    "M": "Kicking (Soccer Ball)",
    "O": "Playing Catch w/Tennis Ball",
    "P": "Dribbling (Basketball)",
    "Q": "Writing",
    "R": "Clapping",
    "S": "Folding Clothes",
}

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import numpy as np

classes = np.unique(y_train_phone)
y_train_bin = label_binarize(y_train_phone, classes=classes)
n_classes = len(classes)

numeric_to_name = {i: ACTIVITIES[le.inverse_transform([i])[0]] for i in classes}

plt.figure(figsize=(14, 10))

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(
        y_train_bin[:, i], y_train_probs[:, i]  # type: ignore
    )
    ap_score = average_precision_score(y_train_bin[:, i], y_train_probs[:, i])  # type: ignore

    label_name = numeric_to_name[i]
    plt.plot(recall, precision, lw=2, label=f"{label_name} (AP = {ap_score:.2f})")

plt.xlabel("Recall", fontsize=14)
plt.ylabel("Precision", fontsize=14)
plt.title(
    "Precision-Recall Curve - Phone Data (Cross-Validation)",
    fontsize=16,
    fontweight="bold",
)
plt.legend(
    loc="lower left", fontsize=10, ncol=2
)  # ncol=2 makes it a grid so it doesn't overlap
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, val_scores = learning_curve(  # type: ignore
    model,
    X_train_phone,
    y_train_phone,
    cv=3,
    scoring="f1_macro",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1,
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(12, 8))

plt.plot(train_sizes, train_mean, "o-", color="blue", label="Training Score", lw=2)
plt.fill_between(
    train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color="blue"
)

plt.plot(train_sizes, val_mean, "o-", color="red", label="Cross-Validation Score", lw=2)
plt.fill_between(
    train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color="red"
)

plt.xlabel(
    "Number of Training Samples",
)
plt.ylabel(
    "F1-Macro Score",
)
plt.title(f"Learning Curve - Phone Data ({len(np.unique(y_train_phone))} Activities)")
plt.legend(
    loc="best",
)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
model_watch = RandomForestClassifier(
    n_estimators=450,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    criterion="entropy",
    n_jobs=-1,
)

score = cross_val_score(
    model,
    X_train_watch,
    y_train_watch,
    cv=3,
    scoring=make_scorer(f1_score, average="macro"),
    n_jobs=-1,
).mean()

score

In [ ]:
from sklearn.model_selection import cross_val_predict

y_preds_watch = cross_val_predict(
    model,
    X_train_watch,
    y_train_watch,
    cv=3,
    n_jobs=-1,
)

In [ ]:
print(classification_report(y_train_watch, y_preds_watch))


cm = confusion_matrix(y_train_watch, y_preds_watch)

fig, ax = plt.subplots(figsize=(14, 12))

# Plot the matrix using the display class
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, cmap="Blues", values_format="d")

disp.ax_.set_xticklabels(
    disp.ax_.get_xticklabels(),
    rotation=45,
    ha="right",
    fontsize=14,
)

disp.ax_.set_yticklabels(disp.ax_.get_yticklabels(), fontsize=14)

disp.ax_.set_title(
    "Confusion Matrix - Phone Sensor (Cross-Validation)", fontsize=18, fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
y_train_probs = cross_val_predict(
    model,
    X_train_watch,
    y_train_watch,
    cv=3,
    method="predict_proba",
    n_jobs=-1,
)

In [ ]:
classes = np.unique(y_train_watch)
y_train_bin = label_binarize(y_train_watch, classes=classes)
n_classes = len(classes)

numeric_to_name = {i: ACTIVITIES[le.inverse_transform([i])[0]] for i in classes}

plt.figure(figsize=(14, 10))

for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(
        y_train_bin[:, i], y_train_probs[:, i]  # type: ignore
    )
    ap_score = average_precision_score(y_train_bin[:, i], y_train_probs[:, i])  # type: ignore

    label_name = numeric_to_name[i]
    plt.plot(recall, precision, lw=2, label=f"{label_name} (AP = {ap_score:.2f})")

plt.xlabel("Recall", fontsize=14)
plt.ylabel("Precision", fontsize=14)
plt.title(
    "Precision-Recall Curve - Phone Data (Cross-Validation)",
    fontsize=16,
    fontweight="bold",
)
plt.legend(
    loc="lower left", fontsize=10, ncol=2
)  # ncol=2 makes it a grid so it doesn't overlap
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)

    max_depth = trial.suggest_int("max_depth", 5, 50)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 5)

    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    criterion = trial.suggest_categorical("criterion", ["entropy", "gini"])

    max_features = trial.suggest_categorical(
        "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]
    )

    max_samples = trial.suggest_categorical("max_samples", [0.9, 1])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        max_features=max_features,
        max_samples=max_samples,
        bootstrap=True,
        n_jobs=-1,
        random_state=42,
    )

    score = cross_val_score(
        model,
        X_train_watch,
        y_train_watch,
        cv=3,
        scoring=make_scorer(f1_score, average="macro"),
        n_jobs=-1,
    ).mean()

    return score


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(),
)


study.optimize(objective, n_trials=50)